# Baseline notebook: Customer churn (Task 02)
Responsible baseline modeling on the small labeled churn dataset. **Decision:** which customers get a retention outreach in the next 14 days. **Target:** `churned` (1 = left within the observation window). **Unit:** one customer.

In [1]:
import pandas as pd, numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, brier_score_loss
df = pd.read_csv('customer-churn-training.csv')
df

,customer_id,tenure_months,support_tickets,monthly_spend_inr,last_login_days,plan_type,churned
0,C001,2,4,499,21,Basic,1
1,C002,18,1,1299,2,Pro,0
2,C003,6,3,799,14,Basic,1
3,C004,30,0,1499,1,Pro,0
4,C005,11,2,999,5,Standard,0
5,C006,1,5,499,30,Basic,1
6,C007,24,1,999,3,Standard,0
7,C008,8,4,799,17,Basic,1
8,C009,15,2,1299,4,Pro,0
9,C010,4,3,499,12,Basic,1


## 1. Data quality checks

In [2]:
print('rows, cols:', df.shape)
print('missing values per column:'); print(df.isna().sum().to_string())
print('duplicate rows:', df.duplicated().sum(), '| duplicate ids:', df.customer_id.duplicated().sum())
print('class balance:'); print(df.churned.value_counts().rename({0:'stayed',1:'churned'}).to_string())
df.describe().T

rows, cols: (12, 7)
missing values per column:
customer_id          0
tenure_months        0
support_tickets      0
monthly_spend_inr    0
last_login_days      0
plan_type            0
churned              0
duplicate rows: 0 | duplicate ids: 0
class balance:
churned
stayed     7
churned    5


,count,mean,std,min,25%,50%,75%,max
tenure_months,12.0,12.166667,9.242524,1.0,5.50,9.5,18.50,30.0
support_tickets,12.0,2.250000,1.602555,0.0,1.00,2.0,3.25,5.0
monthly_spend_inr,12.0,915.666667,332.574895,499.0,724.00,899.0,1074.00,1499.0
last_login_days,12.0,10.000000,9.095453,1.0,2.75,7.0,14.75,30.0
churned,12.0,0.416667,0.514929,0.0,0.00,0.0,1.00,1.0


## 2. Leakage and proxy audit
- `customer_id`: identifier only; **excluded** from features.
- `last_login_days`: only valid if measured **before** the churn window starts. If it is measured after a customer has already left, it is leakage (a churned customer naturally has not logged in). Must be confirmed with the data owner.
- `support_tickets`, `tenure_months`: safe if counted up to the prediction date.
- `monthly_spend_inr` and `plan_type`: nearly the same information (Basic=499-799, Standard=799-999, Pro=999-1499), so they are redundant proxies of each other and of customer income.
- No gender, age, location or religion columns exist, but spend and plan can proxy for affordability.

In [3]:
print(df.groupby('plan_type').churned.agg(n='count', churn_rate='mean'))
print()
print(df.groupby('churned')[['tenure_months','support_tickets','monthly_spend_inr','last_login_days']].mean().round(1))
print('\nplan_type perfectly separates the labels here (Basic = all churn). With n=12 that is a warning sign, not a finding.')

           n  churn_rate
plan_type               
Basic      5         1.0
Pro        3         0.0
Standard   4         0.0

         tenure_months  support_tickets  monthly_spend_inr  last_login_days
churned                                                                    
0                 17.9              1.1             1127.6              3.7
1                  4.2              3.8              619.0             18.8

plan_type perfectly separates the labels here (Basic = all churn). With n=12 that is a warning sign, not a finding.


## 3. Non-ML baseline (rule)
Rule: flag a customer for outreach if `last_login_days >= 10` **or** `plan_type == 'Basic'`. This is simple, explainable, and needs no training.

In [4]:
rule = ((df.last_login_days >= 10) | (df.plan_type=='Basic')).astype(int)
def report(y, p, name):
    tn, fp, fn, tp = confusion_matrix(y, p, labels=[0,1]).ravel()
    print(f'{name}: TP={tp} FP={fp} FN={fn} TN={tn} | precision={tp/max(tp+fp,1):.2f} recall={tp/max(tp+fn,1):.2f}')
    return fp, fn
fp_r, fn_r = report(df.churned, rule, 'Rule baseline (training data)')

Rule baseline (training data): TP=5 FP=0 FN=0 TN=7 | precision=1.00 recall=1.00


## 4. Simple model: logistic regression
Evaluated with **leave-one-out cross-validation** (the only honest option with 12 rows). Features exclude `customer_id` and `plan_type` (to avoid the perfect-separation proxy).

In [5]:
X = df[['tenure_months','support_tickets','monthly_spend_inr','last_login_days']]
y = df.churned
model = make_pipeline(StandardScaler(), LogisticRegression(C=0.5))
proba = cross_val_predict(model, X, y, cv=LeaveOneOut(), method='predict_proba')[:,1]
pred = (proba >= 0.5).astype(int)
fp_m, fn_m = report(y, pred, 'Logistic regression (LOOCV)')
print('Brier score (lower is better):', round(brier_score_loss(y, proba),3))
print('Predicted churn probabilities:', np.round(proba,2))

Logistic regression (LOOCV): TP=5 FP=1 FN=0 TN=6 | precision=0.83 recall=1.00
Brier score (lower is better): 0.073
Predicted churn probabilities: [0.87 0.08 0.54 0.01 0.3  0.98 0.11 0.67 0.16 0.65 0.08 0.52]


## 5. Business metrics with explicit error costs
**Assumptions (to be replaced with real finance numbers):**
- False negative (missed churner): lose about 6 months of average spend, roughly **INR 4,800**.
- False positive (needless outreach/discount): about **INR 150**.
- True positive: outreach saves the customer only some of the time; assume 30% save rate.

In [6]:
COST_FN, COST_FP, SAVE_RATE = 4800, 150, 0.30
def expected_cost(fp, fn, tp):
    return fp*COST_FP + fn*COST_FN + tp*(1-SAVE_RATE)*COST_FN
n_churn = int(y.sum())
no_action = n_churn*COST_FN
for name,(fp,fn) in {'No action':(0,n_churn),'Rule baseline':(fp_r,fn_r),'Logistic (LOOCV)':(fp_m,fn_m)}.items():
    tp = n_churn - fn
    print(f'{name:18s} FP={fp} FN={fn} -> expected cost INR {expected_cost(fp,fn,tp):,.0f}')
print('\nModel metrics to track: recall on churners (primary, FN is costly), precision, Brier score; business metric: net retention savings.')

No action          FP=0 FN=5 -> expected cost INR 24,000
Rule baseline      FP=0 FN=0 -> expected cost INR 16,800
Logistic (LOOCV)   FP=1 FN=0 -> expected cost INR 16,950

Model metrics to track: recall on churners (primary, FN is costly), precision, Brier score; business metric: net retention savings.


## 6. Conclusion
On 12 labeled rows a transparent rule matches or beats the model, and any metric has huge uncertainty. **ML is not justified yet.** Use the rule, collect more data (hundreds of labeled customers with time-stamped features), and re-test the model against this baseline.